In [11]:
# Cell 1: Imports and setup
import os, sys, json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import *
from utils import set_seed, generate_persian_reasoning
from data_loader import OHLCVDataset, create_dataloader
from smc_engine import smc_analysis, smc_feature_vector
from transformer import OHLCVTransformer, ViT, TimeframeFusion
from model import MarketPredictor
from backtester import Backtester
from trading_engine import LocalTradingEngine

set_seed(42)
print(f"Using device: {DEVICE}")

Using device: cpu


In [12]:
# Cell 2: Generate synthetic data
def generate_synthetic_ohlcv(n=2000):
    np.random.seed(0)
    close = 50000 + np.cumsum(np.random.randn(n) * 200)
    high = close + np.abs(np.random.randn(n) * 100)
    low = close - np.abs(np.random.randn(n) * 100)
    open_ = close - np.random.randn(n) * 50
    df = pd.DataFrame({'open': open_, 'high': high, 'low': low, 'close': close})
    return df

df = generate_synthetic_ohlcv(2000)
df.to_csv('synthetic_btc.csv', index=False)
print("Data saved")
df.head()

Data saved


,open,high,low,close
0,50250.683657,50506.102575,50291.475553,50352.810469
1,50478.814970,50604.038927,50248.471913,50432.841911
2,50622.856006,50633.203014,50601.480409,50628.589508
3,51083.639333,51172.605596,50963.123385,51076.768148
4,51382.003399,51458.360907,51276.446559,51450.279746


In [13]:
# Cell 3: Create dataloaders
train_loader = create_dataloader('synthetic_btc.csv', batch_size=32, train=True)
test_loader = create_dataloader('synthetic_btc.csv', batch_size=32, train=False)
sample_x, sample_y = next(iter(train_loader))
print(sample_x.shape, sample_y.shape)

torch.Size([32, 64, 4]) torch.Size([32])


In [ ]:
# Cell 4: Train model
model = MarketPredictor(use_image=False).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()
model.train()
losses = []
for epoch in range(EPOCHS):
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ohlcv_dict = {tf: x for tf in TIMEFRAMES}
        smc_feats = []
        for i in range(x.size(0)):
            seq_df = pd.DataFrame(x[i].cpu().numpy(), columns=['open','high','low','close'])
            smc_res = smc_analysis(seq_df)
            smc_feats.append(smc_feature_vector(smc_res, SEQ_LEN))
        smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)
        logits, confidence = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss/len(train_loader))
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {losses[-1]:.4f}")
torch.save(model.state_dict(), 'market_model.pt')

Epoch 1/20, Loss: 0.7579
Epoch 2/20, Loss: 0.7092
Epoch 3/20, Loss: 0.7062
Epoch 4/20, Loss: 0.7072
Epoch 5/20, Loss: 0.7032
Epoch 6/20, Loss: 0.7015
Epoch 7/20, Loss: 0.7022
Epoch 8/20, Loss: 0.6978
Epoch 9/20, Loss: 0.6951


In [ ]:
# Cell 5: Evaluate
model.eval()
correct = 0; total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ohlcv_dict = {tf: x for tf in TIMEFRAMES}
        smc_feats = []
        for i in range(x.size(0)):
            seq_df = pd.DataFrame(x[i].cpu().numpy(), columns=['open','high','low','close'])
            smc_res = smc_analysis(seq_df)
            smc_feats.append(smc_feature_vector(smc_res, SEQ_LEN))
        smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)
        logits, _ = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
print(f"Test Accuracy: {correct/total:.3f}")

In [ ]:
# Cell 6: Persian reasoning example
model.eval()
x, y = next(iter(test_loader))
x, y = x.to(DEVICE), y.to(DEVICE)
ohlcv_dict = {tf: x for tf in TIMEFRAMES}
smc_feats = []
for i in range(x.size(0)):
    seq_df = pd.DataFrame(x[i].cpu().numpy(), columns=['open','high','low','close'])
    smc_res = smc_analysis(seq_df)
    smc_feats.append(smc_feature_vector(smc_res, SEQ_LEN))
smc_feats = torch.tensor(smc_feats, dtype=torch.float32).to(DEVICE)
logits, confidence = model(ohlcv_dict=ohlcv_dict, smc_features=smc_feats)
probs = torch.softmax(logits, dim=-1)
pred_class = torch.argmax(probs, dim=1)
classes = ['LONG','SHORT','NO_TRADE']
idx = 0
seq_df = pd.DataFrame(x[idx].cpu().numpy(), columns=['open','high','low','close'])
smc_info = smc_analysis(seq_df)
trend = "bullish" if pred_class[idx]==0 else "bearish"
reason = generate_persian_reasoning(classes[pred_class[idx].item()], confidence[idx].item(), smc_info, trend)
print(reason)

In [ ]:
# Cell 7: Run spot backtest with $1000
backtester = Backtester(df, model, initial_capital=1000.0)
metrics = backtester.run()
print(json.dumps(metrics, indent=2))
# Plot equity
plt.plot(backtester.portfolio.equity_curve)
plt.title("Equity Curve ($1000 start)")
plt.show()

In [ ]:
# Cell 8: To launch UI from notebook (optional)
# Uncomment to run the PyQt5 app – it will block the notebook
# from zo import TradingApp
# import sys
# app = QtWidgets.QApplication(sys.argv)
# window = TradingApp()
# window.show()
# sys.exit(app.exec_())